<div style="background: linear-gradient(135deg, #1F3864 0%, #2E5FA3 100%); padding: 48px 40px; border-radius: 12px; margin-bottom: 8px;">
  <h1 style="color: white; font-size: 2.4em; font-weight: 800; margin: 0 0 8px 0; letter-spacing: -0.5px;">Deep Learning for Business Analytics</h1>
  <h2 style="color: #a8c4e8; font-size: 1.3em; font-weight: 400; margin: 0 0 16px 0; font-style: italic;">From Basics to Large Language Models</h2>
  <p style="color: #c8ddf5; font-size: 0.95em; margin: 0 0 24px 0;">Dr. M. Ramasubramaniam &amp; Mr. Daniel Peter</p>
  <div style="background: rgba(255,255,255,0.12); border-radius: 8px; padding: 16px 20px; display: inline-block;">
    <span style="color: #ddeeff; font-size: 1.05em; font-weight: 600;">&#128214; Chapter 6 &nbsp;&middot;&nbsp; Recurrent Neural Networks and LSTMs</span>
  </div>
</div>
<div style="background: #f0f4fa; border-left: 5px solid #2E5FA3; padding: 14px 20px; border-radius: 0 8px 8px 0; margin-top: 4px; color: #333; font-size: 0.97em;">
  <em>How neural networks read sequences — the hidden state mechanism, why vanilla RNNs fail on long-range patterns, how LSTMs fix this with gating, and two end-to-end business applications: demand forecasting and sentiment analysis.</em>
</div>

## What This Chapter Covers

| Section | Topics |
|---------|--------|
| 6.1 Why Sequences Are Different | Order matters · Business sequential data · The hidden state concept |
| 6.2 Vanilla RNN | Unrolling through time · The vanishing gradient problem · `nn.RNN` |
| 6.3 LSTM | Forget, input, output gates · Cell state vs. hidden state · `nn.LSTM` |
| 6.4 Business Application: Demand Forecasting | Kaggle Store Sales · Sliding window preparation · MAE and MAPE · Forecast visualisation |
| 6.5 Bonus: Sentiment Analysis | Text as a token sequence · Word embeddings · Binary LSTM classifier · Bag-of-words baseline |
| 6.6 ★ Bonus: Autoencoders | What an autoencoder is · Encoder–bottleneck–decoder · Anomaly detection · FFN and LSTM variants |
| 6.7 Pipeline: Saving and Deploying the Forecast Model | `ModelPipeline` for sequence models · Window config · Validate input shape · Load and forecast · Retrain with new sales data |

> **Datasets used in this chapter:**
> - **NYC 311 Service Requests** — loaded via NYC Open Data Socrata API (free, public, no account required)
> - **IMDB Movie Reviews** — loaded via `torchtext` / Hugging Face in Section 6.5 (no Kaggle token required)

---
## Setup

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Chapter 6 — Setup
# Run this cell first. It installs all required packages for this session.
# ─────────────────────────────────────────────────────────────────────────────

!pip install -q torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cpu
!pip install -q numpy pandas matplotlib scikit-learn dill datasets kaggle

import copy
import datetime
import getpass
import json
import os
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, TensorDataset

from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import classification_report

# ── Reproducibility ──────────────────────────────────────────────────────────
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'PyTorch {torch.__version__}  |  device: {device}')
print('Setup complete.')

---
## Shared Training Helpers

The three helpers below — `train_one_epoch`, `evaluate`, `run_training`, and `EarlyStopping` — are identical to the versions introduced in Chapter 3 and reused in Chapters 4 and 5. They are reproduced here so this notebook is fully self-contained.

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Shared helpers — identical contract to Chapters 3, 4, and 5.
# ─────────────────────────────────────────────────────────────────────────────

def train_one_epoch(model, loader, criterion, optimiser, device):
    model.train()
    total = 0.0
    for X_batch, y_batch in loader:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)
        optimiser.zero_grad()
        loss = criterion(model(X_batch), y_batch)
        loss.backward()
        # Gradient clipping is good practice for RNNs and LSTMs
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimiser.step()
        total += loss.item()
    return total / len(loader)


def evaluate(model, loader, criterion, device):
    model.eval()
    total = 0.0
    with torch.no_grad():
        for X_batch, y_batch in loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            total += criterion(model(X_batch), y_batch).item()
    return total / len(loader)


def run_training(model, train_loader, val_loader, criterion, optimiser,
                 num_epochs, device, scheduler=None, val_loss_scheduler=None,
                 early_stopping=None, verbose_every=5):
    """Standard training loop. Returns (train_losses, val_losses).

    scheduler          : epoch-based scheduler (StepLR, CosineAnnealingLR)
    val_loss_scheduler : loss-based scheduler (ReduceLROnPlateau)
    early_stopping     : EarlyStopping instance
    """
    train_losses, val_losses = [], []
    for epoch in range(num_epochs):
        tl = train_one_epoch(model, train_loader, criterion, optimiser, device)
        vl = evaluate(model, val_loader, criterion, device)
        train_losses.append(tl)
        val_losses.append(vl)
        if scheduler:
            scheduler.step()
        if val_loss_scheduler:
            val_loss_scheduler.step(vl)
        if verbose_every and (epoch % verbose_every == 0 or epoch == num_epochs - 1):
            print(f'  Epoch {epoch:3d}: train={tl:.4f}  val={vl:.4f}')
        if early_stopping and early_stopping.step(vl, model):
            print(f'  Early stopping at epoch {epoch}  (best val: {early_stopping.best_loss:.4f})')
            early_stopping.restore_best(model)
            break
    return train_losses, val_losses


class EarlyStopping:
    """Halt training when validation loss stops improving. Identical to Chapters 3, 4, and 5."""
    def __init__(self, patience=8, min_delta=1e-4):
        self.patience     = patience
        self.min_delta    = min_delta
        self.best_loss    = float('inf')
        self.counter      = 0
        self.best_weights = None

    def step(self, val_loss, model):
        if val_loss < self.best_loss - self.min_delta:
            self.best_loss    = val_loss
            self.counter      = 0
            self.best_weights = copy.deepcopy(model.state_dict())
        else:
            self.counter += 1
        return self.counter >= self.patience

    def restore_best(self, model):
        if self.best_weights is not None:
            model.load_state_dict(self.best_weights)


print('Helpers defined: train_one_epoch | evaluate | run_training | EarlyStopping')
print('Note: train_one_epoch includes gradient clipping (max_norm=1.0) — important for RNNs.')

---
# 6.1 Why Sequences Are Different

## Order matters

Every architecture covered so far — FFNs in Chapter 4, CNNs in Chapter 5 — treats each input as a self-contained snapshot. Feed a customer record to an FFN and it produces a prediction without knowing whether that customer was acquired last month or ten years ago. Feed a product image to a CNN and it detects defects without knowing that the previous three images on the same production line were also defective.

This is the right design when the inputs are truly independent. But many of the most valuable business datasets are not independent — they are **sequences**: ordered streams of information where what came before directly shapes what comes next.

Consider three examples from business practice:

- **Daily store sales.** Tomorrow's demand depends on yesterday's sales, the seasonality of the week, and whether a promotion is running. A model that ignores the recent sales history will miss systematic patterns the sequence makes obvious.
- **Customer support conversations.** Whether a message is a complaint, a question, or a request depends heavily on the messages that preceded it. Classifying each message in isolation loses the conversational context.
- **Website clickstreams.** Which page a user visits next is strongly influenced by the path they have taken so far. A recommendation engine that ignores order will miss the difference between a customer in early exploration mode and one close to purchase.

**Table 6.1 — Sequential business data and why order matters**

| Data Type | Example | Why order matters |
|-----------|---------|-------------------|
| **Time series** | Daily sales, stock prices, energy usage | Trend, seasonality, and momentum are temporal patterns |
| **Text** | Reviews, contracts, support tickets | Meaning depends on sentence structure and preceding words |
| **Event logs** | Clickstreams, transaction sequences, IoT sensor streams | Context from prior events changes interpretation of the current one |
| **Audio** | Call recordings, quality sensor data | Signal at time *t* encodes information about what just happened |

## The hidden state concept

To process a sequence, a network needs a mechanism for carrying information forward through time. In a CNN, spatial information is preserved through the convolution operation — the kernel sees neighbouring pixels simultaneously. For sequences, the equivalent mechanism is the **hidden state**: a vector that is updated at each step and passed to the next step, acting as a compressed memory of everything the network has seen so far.

Think of it as a rolling summary. After reading step 1, the network produces a hidden state $h_1$ — a compact representation of step 1. After reading step 2 together with $h_1$, it produces $h_2$ — a representation of steps 1 and 2 combined. By the time the network reaches the final step, the hidden state encodes the entire sequence.

This is the core idea behind all recurrent architectures. The specific mechanism for computing and updating the hidden state is what distinguishes a vanilla RNN from an LSTM.

### 📝 Exercise 6.1 — Sequence Problems in Your Domain

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Exercise 6.1
#
# List three sequence-based problems from your own industry or area of interest.
# For each, answer:
#   (a) What is the sequence? (what is each step?)
#   (b) What is the target? (what are you trying to predict?)
#   (c) Why would order matter? (what patterns live in the sequence?)
#
# Problem 1:
#   (a) Sequence: ?
#   (b) Target  : ?
#   (c) Why     : ?
#
# Problem 2:
#   (a) Sequence: ?
#   (b) Target  : ?
#   (c) Why     : ?
#
# Problem 3:
#   (a) Sequence: ?
#   (b) Target  : ?
#   (c) Why     : ?
# ─────────────────────────────────────────────────────────────────────────────

print('Exercise 6.1 complete.')

---
# 6.2 Vanilla RNN

## Unrolling through time

A **vanilla Recurrent Neural Network (RNN)** processes a sequence one step at a time. At each step $t$, it receives two inputs: the current element of the sequence $x_t$ and the hidden state from the previous step $h_{t-1}$. It combines them through a single weight matrix and an activation function to produce the new hidden state $h_t$:

$$h_t = \tanh(W_h \, h_{t-1} + W_x \, x_t + b)$$

The weight matrices $W_h$ and $W_x$ are **shared across all time steps**. The same transformation is applied at every position in the sequence — just as a CNN's convolutional kernel is shared across all spatial positions. This parameter sharing is what makes an RNN tractable regardless of sequence length.

**Figure 6.1** shows this graphically. When we "unroll" an RNN we draw a separate copy of the network for each time step — each copy uses the same weights but receives a different $x_t$ and $h_{t-1}$.

**Table 6.2a — RNN components**

| Symbol | Name | What it is |
|--------|------|------------|
| $x_t$ | Input at step $t$ | One element of the sequence (e.g., one day's sales, one word token) |
| $h_t$ | Hidden state at step $t$ | Compressed memory of the sequence up to and including step $t$ |
| $W_x$ | Input weight matrix | Projects the current input into the hidden space |
| $W_h$ | Recurrent weight matrix | Projects the previous hidden state forward |
| $b$ | Bias | Standard additive offset |
| $\tanh$ | Activation | Keeps hidden state values in $(-1, 1)$ — zero-centred (see Table 3.6) |

## The vanishing gradient problem in RNNs

Chapter 3 introduced the vanishing gradient problem in the context of deep feedforward networks: when gradients are multiplied through many layers, they can shrink exponentially and early layers stop learning. In an RNN, the same problem appears in the **time** dimension instead of the depth dimension.

When backpropagation flows through an unrolled RNN, it multiplies the gradient by $W_h$ at every time step. If the largest singular value of $W_h$ is less than 1, the gradient shrinks by that factor at each step. A sequence of length 100 multiplies the gradient by this factor 100 times — and the result approaches zero. The network cannot adjust weights that produced predictions far back in time.

**The practical consequence:** vanilla RNNs fail on sequences that require remembering information over more than roughly 10–20 steps. For business time series — weekly forecasts drawn from a year of history, or sentiment analysis across a full review — the relevant context typically spans far more steps than a vanilla RNN can reliably use.

The code below makes this concrete. We first train an RNN on a short sine wave (10 steps) — it succeeds. Then on a longer one (60 steps) — the gradient vanishes and it fails.

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# 6.2a — Build a toy sine wave dataset
#
# Each sample is a window of length SEQ_LEN; the target is the next value.
# We test two window lengths to demonstrate the vanishing gradient problem.
# ─────────────────────────────────────────────────────────────────────────────

def make_sine_dataset(seq_len, n_samples=2000, noise=0.02):
    """Generate (X, y) pairs from a noisy sine wave.
    X shape: (n_samples, seq_len, 1)   — one feature per time step
    y shape: (n_samples, 1)            — next value to predict
    """
    t = np.linspace(0, 8 * np.pi, n_samples + seq_len + 1)
    s = np.sin(t) + noise * np.random.randn(len(t))
    X = np.stack([s[i : i + seq_len] for i in range(n_samples)], axis=0)
    y = s[seq_len : seq_len + n_samples].reshape(-1, 1)
    return (
        torch.tensor(X, dtype=torch.float32).unsqueeze(-1),   # (N, L, 1)
        torch.tensor(y, dtype=torch.float32),                 # (N, 1)
    )


def make_loaders(X, y, val_split=0.15, batch_size=64):
    n = len(X)
    split = int(n * (1 - val_split))
    ds_tr = TensorDataset(X[:split], y[:split])
    ds_va = TensorDataset(X[split:], y[split:])
    return (
        DataLoader(ds_tr, batch_size=batch_size, shuffle=True),
        DataLoader(ds_va, batch_size=batch_size, shuffle=False),
    )


SHORT_LEN = 10
LONG_LEN  = 60

X_short, y_short = make_sine_dataset(SHORT_LEN)
X_long,  y_long  = make_sine_dataset(LONG_LEN)

short_train_loader, short_val_loader = make_loaders(X_short, y_short)
long_train_loader,  long_val_loader  = make_loaders(X_long,  y_long)

print(f'Short-sequence dataset:  X={X_short.shape}  y={y_short.shape}')
print(f'Long-sequence  dataset:  X={X_long.shape}   y={y_long.shape}')

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# 6.2b — Vanilla RNN model
#
# nn.RNN returns:
#   output : (batch, seq_len, hidden_size) — hidden state at every step
#   h_n    : (num_layers, batch, hidden_size) — hidden state at the LAST step
#
# For forecasting we only need the last step's hidden state, which we project
# to a single output value through a linear layer.
# ─────────────────────────────────────────────────────────────────────────────

class VanillaRNN(nn.Module):
    """Single-layer vanilla RNN for sequence-to-one regression."""
    def __init__(self, input_size=1, hidden_size=32, num_layers=1, dropout=0.0):
        super().__init__()
        self.rnn = nn.RNN(
            input_size  = input_size,
            hidden_size = hidden_size,
            num_layers  = num_layers,
            batch_first = True,    # input shape: (batch, seq_len, input_size)
            dropout     = dropout if num_layers > 1 else 0.0,
            nonlinearity = 'tanh', # default; 'relu' is also valid
        )
        self.fc = nn.Linear(hidden_size, 1)

    def forward(self, x):
        # x: (batch, seq_len, input_size)
        _, h_n = self.rnn(x)           # h_n: (num_layers, batch, hidden_size)
        out = self.fc(h_n[-1])         # use last layer's final hidden state
        return out                     # (batch, 1)


rnn_short = VanillaRNN(hidden_size=32).to(device)
rnn_long  = VanillaRNN(hidden_size=32).to(device)

print(rnn_short)
n_params = sum(p.numel() for p in rnn_short.parameters())
print(f'\nTrainable parameters: {n_params:,}')

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# 6.2c — Train both RNNs and compare convergence
#
# Figure 6.1 — RNN loss curves: short vs long sequences
# ─────────────────────────────────────────────────────────────────────────────

NUM_EPOCHS_RNN = 40
crit = nn.MSELoss()

torch.manual_seed(SEED)
print('Training RNN on SHORT sequences (seq_len=10):')
rnn_short_train, rnn_short_val = run_training(
    rnn_short, short_train_loader, short_val_loader, crit,
    optim.Adam(rnn_short.parameters(), lr=1e-3),
    num_epochs=NUM_EPOCHS_RNN, device=device, verbose_every=10,
)

torch.manual_seed(SEED)
print('\nTraining RNN on LONG sequences (seq_len=60):')
rnn_long_train, rnn_long_val = run_training(
    rnn_long, long_train_loader, long_val_loader, crit,
    optim.Adam(rnn_long.parameters(), lr=1e-3),
    num_epochs=NUM_EPOCHS_RNN, device=device, verbose_every=10,
)

# ── Figure 6.1 ────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
fig.patch.set_facecolor('white')

for ax, train_h, val_h, title, color in zip(
    axes,
    [rnn_short_train, rnn_long_train],
    [rnn_short_val,   rnn_long_val],
    [f'Vanilla RNN  (seq_len={SHORT_LEN})',
     f'Vanilla RNN  (seq_len={LONG_LEN})'],
    ['#27ae60', '#e74c3c'],
):
    ax.set_facecolor('white')
    ep = range(1, len(train_h) + 1)
    ax.plot(ep, train_h, color=color,     lw=2.0, label='Train loss')
    ax.plot(ep, val_h,   color='#555555', lw=2.0, label='Val loss',  linestyle='--')
    ax.set_xlabel('Epoch', fontsize=11)
    ax.set_ylabel('MSE Loss', fontsize=11)
    ax.set_title(title, fontsize=11, fontweight='bold', color='#1F3864')
    ax.legend(fontsize=9)

fig.suptitle('Figure 6.1 — Vanilla RNN: Short Sequences Succeed, Long Sequences Fail',
             fontsize=11, fontweight='bold', color='#1F3864')
plt.tight_layout()
plt.show()

print(f'\nFinal val loss  (seq_len={SHORT_LEN}): {rnn_short_val[-1]:.4f}')
print(f'Final val loss  (seq_len={LONG_LEN}):  {rnn_long_val[-1]:.4f}')
print(f'\nThe long-sequence RNN fails to converge — gradients vanish before reaching the')
print(f'early time steps. Section 6.3 introduces LSTMs, which solve this.')

### 📝 Exercise 6.2 — Observe RNN Failure on Long Sequences

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Exercise 6.2
#
# Task: Train a VanillaRNN on the LONG sequence dataset (seq_len=60) but with
#       a MUCH larger hidden size (hidden_size=128).
#
#       Does increasing hidden size fix the vanishing gradient problem?
#       Print the final validation loss and compare it to the seq_len=10 result.
#
# Answer the following as comments after your code:
#   Q1. Did a larger hidden state help? Why or why not?
#   Q2. Is the vanishing gradient problem a problem of model capacity
#       or of architecture design?
# ─────────────────────────────────────────────────────────────────────────────

torch.manual_seed(SEED)
# rnn_large = VanillaRNN(hidden_size=128).to(device)
# ...

# See the Answer Key at the end of this chapter.
print('Exercise 6.2 complete.')

---
# 6.3 LSTM

## How LSTMs fix the vanishing gradient problem

The **Long Short-Term Memory (LSTM)**, introduced by Hochreiter and Schmidhuber in 1997, solves the vanishing gradient problem through a fundamentally different internal design. Instead of a single hidden state updated through a single $\tanh$ transformation, an LSTM maintains **two** internal vectors at each step:

| Vector | Name | Role |
|--------|------|------|
| $h_t$ | **Hidden state** | Short-term output — passed to the next step and to the output layer |
| $c_t$ | **Cell state** | Long-term memory — flows through the sequence with minimal modification |

The cell state is the key insight. It flows from step to step along a path that is **additively** updated — not multiplicatively transformed by a weight matrix at every step. This means gradients can flow backward through many steps without shrinking, because addition does not change gradient magnitude the way multiplication does.

Three learnable **gates** control what information enters, leaves, or is forgotten from the cell state at each step:

## The three gates

All three gates share the same structural form: a sigmoid activation applied to a linear combination of the current input $x_t$ and previous hidden state $h_{t-1}$. The sigmoid output lies in $(0, 1)$, functioning as a smooth on/off switch.

**Forget gate** — decides what to erase from the cell state:
$$f_t = \sigma(W_f [h_{t-1}, x_t] + b_f)$$
Values near 0 → forget; values near 1 → keep.

**Input gate** — decides what new information to write to the cell state:
$$i_t = \sigma(W_i [h_{t-1}, x_t] + b_i)$$
$$\tilde{c}_t = \tanh(W_c [h_{t-1}, x_t] + b_c)$$
The input gate controls *how much* of the candidate values $\tilde{c}_t$ to add.

**Cell state update** — combines forgetting and writing:
$$c_t = f_t \odot c_{t-1} + i_t \odot \tilde{c}_t$$

**Output gate** — decides what to expose from the cell state as the hidden state:
$$o_t = \sigma(W_o [h_{t-1}, x_t] + b_o)$$
$$h_t = o_t \odot \tanh(c_t)$$

**Table 6.3 — LSTM gate summary**

| Gate | Symbol | Controls | Business analogy |
|------|--------|----------|------------------|
| Forget | $f_t$ | What old memory to discard | "Last year's seasonality is no longer relevant after the product was discontinued" |
| Input | $i_t$ | What new information to record | "This week's promotion data should be remembered" |
| Output | $o_t$ | What memory to use right now | "Surface the promotional uplift when making this week's forecast" |

> **Why does this solve the vanishing gradient problem?** The cell state update $c_t = f_t \odot c_{t-1} + i_t \odot \tilde{c}_t$ is an **additive** update — gradients flowing back through $c_t$ pass through addition and element-wise multiplication by the forget gate, not through a full matrix multiply. When the forget gate is near 1, gradients flow almost unchanged across many steps. The LSTM can learn to keep gradients alive as long as the task requires it.

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# 6.3a — LSTM model
#
# nn.LSTM returns:
#   output : (batch, seq_len, hidden_size)          — hidden state at every step
#   (h_n, c_n) : each (num_layers, batch, hidden_size) — final hidden and cell states
#
# Architecture is identical to VanillaRNN except nn.RNN is replaced by nn.LSTM.
# ─────────────────────────────────────────────────────────────────────────────

class LSTMNet(nn.Module):
    """Single- or multi-layer LSTM for sequence-to-one regression or classification."""
    def __init__(self, input_size=1, hidden_size=64, num_layers=2,
                 dropout=0.2, output_size=1):
        super().__init__()
        self.lstm = nn.LSTM(
            input_size  = input_size,
            hidden_size = hidden_size,
            num_layers  = num_layers,
            batch_first = True,
            dropout     = dropout if num_layers > 1 else 0.0,
        )
        self.dropout = nn.Dropout(dropout)
        self.fc      = nn.Linear(hidden_size, output_size)

    def forward(self, x):
        # x: (batch, seq_len, input_size)
        _, (h_n, _) = self.lstm(x)     # h_n: (num_layers, batch, hidden_size)
        out = self.dropout(h_n[-1])    # last layer, final step
        return self.fc(out)            # (batch, output_size)


lstm_long = LSTMNet(input_size=1, hidden_size=64, num_layers=2, dropout=0.2).to(device)
print(lstm_long)
n_params = sum(p.numel() for p in lstm_long.parameters())
print(f'\nTrainable parameters: {n_params:,}')

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# 6.3b — Train LSTM on the long-sequence dataset and compare to vanilla RNN
#
# Figure 6.2 — LSTM vs vanilla RNN on seq_len=60
# ─────────────────────────────────────────────────────────────────────────────

torch.manual_seed(SEED)
print('Training LSTM on LONG sequences (seq_len=60):')
lstm_long_train, lstm_long_val = run_training(
    lstm_long, long_train_loader, long_val_loader, crit,
    optim.Adam(lstm_long.parameters(), lr=1e-3),
    num_epochs=NUM_EPOCHS_RNN, device=device, verbose_every=10,
)

# ── Figure 6.2 ────────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 4))
fig.patch.set_facecolor('white')
ax.set_facecolor('white')

ep = range(1, NUM_EPOCHS_RNN + 1)
ax.plot(ep, rnn_long_val,  color='#e74c3c', lw=2.0, label='Vanilla RNN (seq_len=60)')
ax.plot(ep, lstm_long_val, color='#2980b9', lw=2.5, label='LSTM        (seq_len=60)')

ax.set_xlabel('Epoch', fontsize=11)
ax.set_ylabel('Validation MSE Loss', fontsize=11)
ax.set_title('Figure 6.2 — LSTM Solves What Vanilla RNN Cannot',
             fontsize=11, fontweight='bold', color='#1F3864')
ax.legend(fontsize=10)
plt.tight_layout()
plt.show()

print(f'\nFinal val loss — Vanilla RNN : {rnn_long_val[-1]:.4f}')
print(f'Final val loss — LSTM        : {lstm_long_val[-1]:.4f}')

### 📝 Exercise 6.3 — Replace RNN with LSTM

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Exercise 6.3
#
# Task: Train an LSTMNet on the SHORT sequence dataset (seq_len=10).
#       Plot its val loss on the same axes as the VanillaRNN short-sequence result.
#
# Answer the following as comments after your code:
#   Q1. On short sequences, is the LSTM meaningfully better than the RNN?
#       What does this tell you about when to use each architecture?
#   Q2. LSTMs have more parameters than RNNs of the same hidden size.
#       Roughly how many more? (compare n_params from 6.2b and 6.3a)
# ─────────────────────────────────────────────────────────────────────────────

torch.manual_seed(SEED)
# lstm_short = LSTMNet(hidden_size=32, num_layers=1, dropout=0.0).to(device)
# ...

# See the Answer Key at the end of this chapter.
print('Exercise 6.3 complete.')

---
# 6.4 Business Application: Demand Forecasting

## The business problem

Accurate demand forecasting sits at the centre of retail operations. Overstock ties up working capital and generates waste. Understock loses sales and damages customer relationships. A model that forecasts the next four weeks of sales for each store and product family — with enough accuracy to drive purchasing decisions — delivers direct and measurable business value.

The **Superstore Sales** dataset from Kaggle (`rohitsahoo/sales-forecasting`) contains 9,994 retail orders placed between January 2014 and December 2017, covering furniture, office supplies, and technology categories across the United States. We aggregate orders to total daily revenue, then train an LSTM to forecast the next 28 days from a sliding window of historical observations.

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# 6.4 — Load NYC 311 daily call volume via Socrata Open Data API
#
# Dataset : NYC 311 Service Requests from 2010 to Present
# Portal  : https://data.cityofnewyork.us/Social-Services/311-Service-Requests-from-2010-to-Present/erm2-nwe9
# API     : Socrata SODA — free, public, no account or token required
# Query   : count of requests per calendar day, 2014-01-01 to 2017-12-31
# ─────────────────────────────────────────────────────────────────────────────

import urllib.request, io

SODA_URL = (
    'https://data.cityofnewyork.us/resource/erm2-nwe9.csv'
    '?$select=date_trunc_ymd(created_date)%20AS%20date'
    ',count(*)%20AS%20n_calls'
    "&$where=created_date%20between%20'2014-01-01T00:00:00'"
    "%20and%20'2017-12-31T23:59:59'"
    '&$group=date_trunc_ymd(created_date)'
    '&$order=date_trunc_ymd(created_date)%20ASC'
    '&$limit=2000'
)

req = urllib.request.Request(SODA_URL, headers={'User-Agent': 'Mozilla/5.0'})
with urllib.request.urlopen(req, timeout=30) as r:
    raw_csv = r.read().decode('utf-8')

daily = (
    pd.read_csv(io.StringIO(raw_csv), parse_dates=['date'])
    .rename(columns={'n_calls': 'total_sales'})
    .sort_values('date')
    .reset_index(drop=True)
)
daily['total_sales'] = daily['total_sales'].astype(float)

# Reindex to a full calendar and fill any missing days with the series median
full_idx = pd.date_range(daily.date.min(), daily.date.max(), freq='D')
daily = (
    daily.set_index('date')
    .reindex(full_idx)
    .rename_axis('date')
    .reset_index()
)
daily['total_sales'] = daily['total_sales'].fillna(daily['total_sales'].median())

print(f'Loaded {len(daily):,} days of NYC 311 call volume.')
print(f'Date range  : {daily.date.min().date()}  →  {daily.date.max().date()}')
print(f'Calls/day   : min={daily.total_sales.min():.0f}  max={daily.total_sales.max():.0f}  mean={daily.total_sales.mean():.0f}')


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# 6.4a — Inspect the series
# ─────────────────────────────────────────────────────────────────────────────

print(daily.describe())
print()
print('Each row = one calendar day.')
print('total_sales = number of 311 service requests received that day.')
print('(Column named total_sales for consistency with preprocessing code below.)')


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# 6.4b — Exploratory data analysis
#
# Figure 6.3 — NYC 311 daily call volume over time
# ─────────────────────────────────────────────────────────────────────────────

fig, axes = plt.subplots(2, 1, figsize=(13, 7))
fig.patch.set_facecolor('white')

axes[0].set_facecolor('white')
axes[0].plot(daily.date, daily.total_sales, color='#2980b9', lw=0.9, alpha=0.85)
axes[0].set_title('(a) NYC 311 Daily Call Volume — 2014 to 2017',
                  fontsize=11, fontweight='bold', color='#1F3864')
axes[0].set_ylabel('Service Requests per Day', fontsize=10)
axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:,.0f}'))

daily['rolling_30'] = daily.total_sales.rolling(30, center=True).mean()
axes[1].set_facecolor('white')
axes[1].plot(daily.date, daily.total_sales, color='#aac4e0', lw=0.7, alpha=0.7, label='Daily')
axes[1].plot(daily.date, daily.rolling_30,  color='#1F3864', lw=2.0, label='30-day rolling mean')
axes[1].set_title('(b) Trend: 30-Day Rolling Mean',
                  fontsize=11, fontweight='bold', color='#1F3864')
axes[1].set_ylabel('Service Requests per Day', fontsize=10)
axes[1].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:,.0f}'))
axes[1].legend(fontsize=9)

fig.suptitle('Figure 6.3 — NYC 311 Call Volume: Clear Trend and Weekly Seasonality',
             fontsize=11, fontweight='bold', color='#1F3864')
plt.tight_layout()
plt.show()


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# 6.4c — Preprocessing: scale and create sliding windows
#
# The LSTM receives sequences of shape (batch, WINDOW_SIZE, n_features).
# Each window of WINDOW_SIZE past values predicts the next FORECAST_HORIZON values.
#
# WINDOW_SIZE     : how many past days the model sees at each step
# FORECAST_HORIZON: how many days ahead we want to predict (28 = 4 weeks)
# ─────────────────────────────────────────────────────────────────────────────

WINDOW_SIZE      = 60    # look-back window (days)
FORECAST_HORIZON = 1     # predict one day at a time; we roll forward for multi-step
VAL_DAYS         = 90    # hold out the final 90 days as validation

# Scale to [0, 1] — MinMaxScaler is common for sales data
# Fit ONLY on the training portion (temporal train/val split)
sales_values = daily.total_sales.values.reshape(-1, 1)
split_idx    = len(sales_values) - VAL_DAYS

scaler = MinMaxScaler(feature_range=(0, 1))
scaler.fit(sales_values[:split_idx])
scaled = scaler.transform(sales_values).flatten()


def make_windows(series, window_size, horizon=1):
    """Slide a window across a 1-D series.
    Returns:
        X : (n_samples, window_size, 1) — input sequences
        y : (n_samples, 1)             — target (next value)
    """
    X, y = [], []
    for i in range(len(series) - window_size - horizon + 1):
        X.append(series[i : i + window_size])
        y.append(series[i + window_size : i + window_size + horizon])
    X = torch.tensor(np.array(X), dtype=torch.float32).unsqueeze(-1)  # (N, W, 1)
    y = torch.tensor(np.array(y), dtype=torch.float32)                 # (N, H)
    return X, y


X_all, y_all = make_windows(scaled, WINDOW_SIZE, FORECAST_HORIZON)

# Temporal split — training samples that end before the val period start
split = split_idx - WINDOW_SIZE
X_tr, y_tr = X_all[:split], y_all[:split]
X_va, y_va = X_all[split:], y_all[split:]

fc_train_loader = DataLoader(TensorDataset(X_tr, y_tr), batch_size=64, shuffle=True)
fc_val_loader   = DataLoader(TensorDataset(X_va, y_va), batch_size=64, shuffle=False)

print(f'Window size      : {WINDOW_SIZE} days')
print(f'Forecast horizon : {FORECAST_HORIZON} day')
print(f'Train windows    : {len(X_tr):,}  |  shape: {X_tr.shape}')
print(f'Val   windows    : {len(X_va):,}  |  shape: {X_va.shape}')

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# 6.4d — Train the demand forecast LSTM
# ─────────────────────────────────────────────────────────────────────────────

FORECAST_CONFIG = dict(
    input_size  = 1,
    hidden_size = 128,
    num_layers  = 2,
    dropout     = 0.2,
    output_size = FORECAST_HORIZON,
)

torch.manual_seed(SEED)
fc_model = LSTMNet(**FORECAST_CONFIG).to(device)

fc_crit  = nn.MSELoss()
fc_opt   = optim.Adam(fc_model.parameters(), lr=1e-3)
fc_sched = optim.lr_scheduler.CosineAnnealingLR(fc_opt, T_max=50)
fc_es    = EarlyStopping(patience=10)

print('Training demand forecast LSTM (up to 50 epochs, CosineAnnealingLR):')
fc_train_hist, fc_val_hist = run_training(
    fc_model, fc_train_loader, fc_val_loader, fc_crit, fc_opt,
    num_epochs=50, device=device,
    scheduler=fc_sched, early_stopping=fc_es, verbose_every=5,
)
print('\nTraining complete.')

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# 6.4e — Evaluate: MAE and MAPE on the validation set
#
# MAE  (Mean Absolute Error)           — average absolute forecast error in units
# MAPE (Mean Absolute Percentage Error) — average % error, scale-independent
#
# Figure 6.4 — Forecast vs actual on validation period
# ─────────────────────────────────────────────────────────────────────────────

fc_model.eval()
preds_scaled, actuals_scaled = [], []

with torch.no_grad():
    for X_b, y_b in fc_val_loader:
        preds_scaled.append(fc_model(X_b.to(device)).cpu().numpy())
        actuals_scaled.append(y_b.numpy())

preds_scaled   = np.concatenate(preds_scaled).flatten()
actuals_scaled = np.concatenate(actuals_scaled).flatten()

# Inverse-transform to original sales units
preds   = scaler.inverse_transform(preds_scaled.reshape(-1, 1)).flatten()
actuals = scaler.inverse_transform(actuals_scaled.reshape(-1, 1)).flatten()

mae  = np.mean(np.abs(preds - actuals))
# Avoid division by zero in MAPE
mask = actuals > 0
mape = np.mean(np.abs((preds[mask] - actuals[mask]) / actuals[mask])) * 100

print(f'Validation MAE  : {mae:>10,.0f} units/day')
print(f'Validation MAPE : {mape:>10.2f}%')

# ── Figure 6.4 ────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(2, 1, figsize=(13, 7))
fig.patch.set_facecolor('white')

# Panel (a): full validation period
axes[0].set_facecolor('white')
axes[0].plot(actuals, color='#1F3864', lw=1.2, label='Actual')
axes[0].plot(preds,   color='#e74c3c', lw=1.2, label='Forecast', alpha=0.85)
axes[0].set_title('(a) Forecast vs Actual — Full Validation Period',
                  fontsize=11, fontweight='bold', color='#1F3864')
axes[0].set_ylabel('Total Units Sold', fontsize=10)
axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:,.0f}'))
axes[0].legend(fontsize=9)

# Panel (b): loss curves
axes[1].set_facecolor('white')
ep = range(1, len(fc_train_hist) + 1)
axes[1].plot(ep, fc_train_hist, color='#2980b9', lw=2.0, label='Train MSE')
axes[1].plot(ep, fc_val_hist,   color='#e67e22', lw=2.0, label='Val MSE',  linestyle='--')
axes[1].set_title('(b) Training Loss Curves',
                  fontsize=11, fontweight='bold', color='#1F3864')
axes[1].set_xlabel('Epoch', fontsize=10)
axes[1].set_ylabel('MSE Loss (scaled)', fontsize=10)
axes[1].legend(fontsize=9)

fig.suptitle(f'Figure 6.4 — Demand Forecast LSTM  |  MAE: {mae:,.0f}  MAPE: {mape:.1f}%',
             fontsize=11, fontweight='bold', color='#1F3864')
plt.tight_layout()
plt.show()

### 📝 Exercise 6.4 — End-to-End Demand Forecast

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Exercise 6.4
#
# Task A: Change WINDOW_SIZE to 30 (instead of 60). Retrain the same LSTM
#         architecture. Does MAE improve or worsen?
#
# Task B: Add a second feature — the day of week (0=Monday ... 6=Sunday).
#         This requires:
#           - Updating make_windows to accept a 2-D series (N_days × n_features)
#           - Changing input_size=2 in FORECAST_CONFIG
#         Does the additional feature improve MAPE?
#
# Task C: Answer the following as comments:
#   Q1. MAPE is often preferred over MAE for demand forecasting. Why?
#       (Hint: think about stores with very different sales volumes.)
#   Q2. We used a temporal train/val split rather than a random split.
#       Why is a random split inappropriate for time series data?
# ─────────────────────────────────────────────────────────────────────────────

# Task A
# WINDOW_SIZE_A = 30
# ...

# Task B
# daily['dow'] = daily.date.dt.dayofweek  # 0 = Monday
# ...

# See the Answer Key at the end of this chapter.
print('Exercise 6.4 complete.')

---
# 6.5 Bonus: Sentiment Analysis

## Text as a token sequence

The second major sequential data type is text. A sentence is a sequence of words — or more precisely, of **tokens** — and the meaning of each token depends on the tokens that preceded it. "The product is not good" and "The product is good" are identical except for one token, but have opposite meanings.

To feed text into a neural network we need two things:

1. **A vocabulary** — a mapping from every unique word to an integer index.
2. **Word embeddings** — a learnable lookup table that maps each integer index to a dense vector of real numbers. Each word is represented not by a single integer but by a vector of `EMBED_DIM` floats, where geometrically close vectors represent semantically similar words.

The network receives a sequence of embedding vectors — one per token — and processes them left-to-right with an LSTM. The final hidden state summarises the whole review and is passed to a classification head.

**Table 6.5 — Text preprocessing for an LSTM classifier**

| Step | What happens | Result |
|------|-------------|--------|
| Tokenise | Split each review into a list of lower-cased words | `['the', 'product', 'is', 'great']` |
| Build vocabulary | Assign integer ID to each unique word | `{'the': 1, 'product': 2, 'is': 3, ...}` |
| Encode | Replace each word with its ID | `[1, 2, 3, 47]` |
| Pad / truncate | Make all sequences the same length | `[1, 2, 3, 47, 0, 0, ...]` (0 = padding) |
| Embed | Lookup table: integer → dense vector | `(batch, seq_len, embed_dim)` |

We use the **IMDB Movie Reviews** dataset — 25,000 training reviews labelled positive or negative — available directly via Hugging Face Datasets.

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# 6.5a — Load IMDB dataset from Hugging Face
#
# No Kaggle token required — datasets downloads directly.
# ─────────────────────────────────────────────────────────────────────────────

from datasets import load_dataset

imdb = load_dataset('imdb')

print(f"Train split : {len(imdb['train']):,} reviews")
print(f"Test  split : {len(imdb['test']):,}  reviews")
print(f"Labels      : 0 = negative  |  1 = positive")
print()
sample_text  = imdb['train'][0]['text']
sample_label = imdb['train'][0]['label']
print(f'Example review (label={sample_label}):')
print(sample_text[:300], '...')

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# 6.5b — Build vocabulary and encode reviews
#
# We keep only the top VOCAB_SIZE most frequent words.
# Out-of-vocabulary tokens are mapped to index 0 (same as padding).
# MAX_LEN controls the fixed sequence length after padding/truncation.
# ─────────────────────────────────────────────────────────────────────────────

from collections import Counter
import re

VOCAB_SIZE = 20_000
MAX_LEN    = 256
PAD_IDX    = 0


def simple_tokenise(text):
    return re.findall(r"[a-z']+", text.lower())


# Build vocabulary from training set only
counter = Counter()
for item in imdb['train']:
    counter.update(simple_tokenise(item['text']))

vocab = {word: idx + 1 for idx, (word, _) in enumerate(counter.most_common(VOCAB_SIZE))}
# index 0 reserved for padding; unknown words map to 0


def encode_and_pad(text, vocab, max_len):
    ids = [vocab.get(w, PAD_IDX) for w in simple_tokenise(text)]
    ids = ids[:max_len]
    ids += [PAD_IDX] * (max_len - len(ids))
    return ids


def build_tensors(split):
    X = torch.tensor(
        [encode_and_pad(item['text'], vocab, MAX_LEN) for item in split],
        dtype=torch.long,
    )
    y = torch.tensor([item['label'] for item in split], dtype=torch.float32)
    return X, y


print('Encoding training set...')
X_imdb_tr, y_imdb_tr = build_tensors(imdb['train'])
print('Encoding test set...')
X_imdb_te, y_imdb_te = build_tensors(imdb['test'])

imdb_train_loader = DataLoader(TensorDataset(X_imdb_tr, y_imdb_tr),
                               batch_size=64, shuffle=True)
imdb_test_loader  = DataLoader(TensorDataset(X_imdb_te, y_imdb_te),
                               batch_size=64, shuffle=False)

print(f'\nVocabulary size : {len(vocab):,}')
print(f'X_train shape   : {X_imdb_tr.shape}  (samples × seq_len)')
print(f'Class balance   : {y_imdb_tr.mean():.0%} positive')

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# 6.5c — Bag-of-words baseline
#
# Before training an LSTM we establish a simple baseline:
# count how often each vocabulary word appears and train a logistic regression.
# If the LSTM can't beat this, something is wrong.
# ─────────────────────────────────────────────────────────────────────────────

from sklearn.linear_model import LogisticRegression
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics import accuracy_score

bow_vectoriser = CountVectorizer(max_features=VOCAB_SIZE, binary=True, lowercase=True)
X_bow_tr = bow_vectoriser.fit_transform([item['text'] for item in imdb['train']])
X_bow_te = bow_vectoriser.transform([item['text'] for item in imdb['test']])

y_bow_tr = np.array([item['label'] for item in imdb['train']])
y_bow_te = np.array([item['label'] for item in imdb['test']])

print('Training bag-of-words logistic regression baseline...')
bow_clf = LogisticRegression(max_iter=1000, C=1.0)
bow_clf.fit(X_bow_tr, y_bow_tr)
bow_acc = accuracy_score(y_bow_te, bow_clf.predict(X_bow_te))
print(f'Bag-of-words baseline accuracy: {bow_acc:.4f} ({bow_acc*100:.1f}%)')

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# 6.5d — LSTM sentiment classifier
#
# nn.Embedding maps token IDs → dense vectors (learned during training).
# padding_idx=0 ensures gradients do not update the embedding for padding tokens.
# Output is a single logit; BCEWithLogitsLoss is numerically stable.
# ─────────────────────────────────────────────────────────────────────────────

class SentimentLSTM(nn.Module):
    """LSTM-based binary sentiment classifier."""
    def __init__(self, vocab_size, embed_dim, hidden_size, num_layers, dropout):
        super().__init__()
        self.embedding = nn.Embedding(
            num_embeddings = vocab_size + 1,  # +1 for padding index 0
            embedding_dim  = embed_dim,
            padding_idx    = PAD_IDX,
        )
        self.lstm = nn.LSTM(
            input_size  = embed_dim,
            hidden_size = hidden_size,
            num_layers  = num_layers,
            batch_first = True,
            dropout     = dropout if num_layers > 1 else 0.0,
        )
        self.dropout = nn.Dropout(dropout)
        self.fc      = nn.Linear(hidden_size, 1)

    def forward(self, x):
        # x: (batch, seq_len)  — integer token IDs
        emb = self.dropout(self.embedding(x))   # (batch, seq_len, embed_dim)
        _, (h_n, _) = self.lstm(emb)            # h_n: (num_layers, batch, hidden)
        out = self.dropout(h_n[-1])             # last layer, final step
        return self.fc(out).squeeze(1)          # (batch,) — raw logit


SENTIMENT_CONFIG = dict(
    vocab_size  = VOCAB_SIZE,
    embed_dim   = 128,
    hidden_size = 128,
    num_layers  = 2,
    dropout     = 0.3,
)

torch.manual_seed(SEED)
sent_model = SentimentLSTM(**SENTIMENT_CONFIG).to(device)

n_params = sum(p.numel() for p in sent_model.parameters())
print(sent_model)
print(f'\nTrainable parameters: {n_params:,}')

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# 6.5e — Train and evaluate the sentiment classifier
#
# BCEWithLogitsLoss combines Sigmoid + Binary Cross-Entropy in one numerically
# stable operation. Targets must be float (0.0 or 1.0).
#
# Figure 6.5 — Sentiment LSTM: loss curves and classification report
# ─────────────────────────────────────────────────────────────────────────────

sent_crit  = nn.BCEWithLogitsLoss()
sent_opt   = optim.Adam(sent_model.parameters(), lr=5e-4)
sent_sched = optim.lr_scheduler.ReduceLROnPlateau(sent_opt, patience=2, factor=0.5)
sent_es    = EarlyStopping(patience=5)

# Use a portion of test set as validation during training
N_VAL = 5000
imdb_val_loader = DataLoader(
    TensorDataset(X_imdb_te[:N_VAL], y_imdb_te[:N_VAL]),
    batch_size=64, shuffle=False,
)
imdb_eval_loader = DataLoader(
    TensorDataset(X_imdb_te[N_VAL:], y_imdb_te[N_VAL:]),
    batch_size=64, shuffle=False,
)

print('Training sentiment LSTM (up to 10 epochs):')
sent_train_hist, sent_val_hist = run_training(
    sent_model, imdb_train_loader, imdb_val_loader, sent_crit, sent_opt,
    num_epochs=10, device=device,
    val_loss_scheduler=sent_sched, early_stopping=sent_es, verbose_every=1,
)
print('\nTraining complete.')

# ── Evaluation ────────────────────────────────────────────────────────────────
sent_model.eval()
all_preds, all_labels = [], []
with torch.no_grad():
    for X_b, y_b in imdb_eval_loader:
        logits = sent_model(X_b.to(device)).cpu()
        preds  = (torch.sigmoid(logits) > 0.5).long()
        all_preds.extend(preds.numpy())
        all_labels.extend(y_b.long().numpy())

lstm_acc = accuracy_score(all_labels, all_preds)

print(f'\nBag-of-words baseline  accuracy: {bow_acc:.4f} ({bow_acc*100:.1f}%)')
print(f'LSTM classifier        accuracy: {lstm_acc:.4f} ({lstm_acc*100:.1f}%)')
print()
print(classification_report(all_labels, all_preds, target_names=['Negative', 'Positive']))

# ── Figure 6.5 ────────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 4))
fig.patch.set_facecolor('white')
ax.set_facecolor('white')
ep = range(1, len(sent_train_hist) + 1)
ax.plot(ep, sent_train_hist, color='#2980b9', lw=2.0, label='Train loss')
ax.plot(ep, sent_val_hist,   color='#e67e22', lw=2.0, label='Val loss',  linestyle='--')
ax.axhline(y=min(sent_val_hist), color='#27ae60', lw=1.2, linestyle=':',
           label=f'Best val loss = {min(sent_val_hist):.4f}')
ax.set_xlabel('Epoch', fontsize=11)
ax.set_ylabel('BCE Loss', fontsize=11)
ax.set_title(f'Figure 6.5 — Sentiment LSTM Loss  |  Test Accuracy: {lstm_acc*100:.1f}%',
             fontsize=11, fontweight='bold', color='#1F3864')
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()

### 📝 Exercise 6.5 — Sentiment Classifier

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Exercise 6.5
#
# Task A: Reduce embed_dim to 32 (from 128). Retrain and compare accuracy.
#         How much does the embedding dimension matter?
#
# Task B: Change MAX_LEN to 512 and retrain. Does a longer context window
#         improve accuracy? Comment on the trade-off in training time.
#
# Task C: Answer the following as comments:
#   Q1. The bag-of-words baseline ignores word order entirely. For which types
#       of reviews would you expect the LSTM to show the biggest advantage?
#       (Give two specific examples.)
#   Q2. We used a learned embedding layer. Alternatively, we could use
#       pre-trained embeddings (e.g., GloVe or word2vec). What is the main
#       advantage of pre-trained embeddings for small datasets?
# ─────────────────────────────────────────────────────────────────────────────

# Task A
# SENTIMENT_CONFIG_A = {**SENTIMENT_CONFIG, 'embed_dim': 32}
# ...

# See the Answer Key at the end of this chapter.
print('Exercise 6.5 complete.')

---
# 6.6 ★ Bonus: Autoencoders

## What is an autoencoder?

An **autoencoder** is a neural network trained to reconstruct its own input. It has no labels — the target is the input itself. This makes it an **unsupervised** architecture, which distinguishes it from every model built so far in this book.

The network is divided into two halves:

| Part | Name | Role |
|------|------|------|
| First half | **Encoder** | Compresses the input into a compact representation called the **latent vector** or **bottleneck** |
| Second half | **Decoder** | Reconstructs the original input from the latent vector |

The bottleneck is the key. Because it has fewer dimensions than the input, the encoder is forced to learn the most important features of the data — anything that cannot be compressed into the bottleneck is discarded.

Training minimises **reconstruction loss** — typically Mean Squared Error between the original input and the decoder's output:

$$\mathcal{L} = \frac{1}{n} \sum_{i=1}^{n} (x_i - \hat{x}_i)^2$$

**Table 6.6 — Autoencoder compared to the architectures covered in this book**

| Architecture | Chapter | Supervision | What it learns |
|-------------|---------|-------------|----------------|
| FFN | 4 | Supervised | Mapping from features to a label |
| CNN | 5 | Supervised | Spatial patterns that predict a class |
| RNN / LSTM | 6 | Supervised | Temporal patterns that predict the next value |
| **Autoencoder** | **6 ★** | **Unsupervised** | **Compact representations that preserve structure** |

## Why autoencoders matter for business

Because the bottleneck forces the network to distil the essential structure of the data, autoencoders are useful whenever the goal is to understand or clean the data rather than predict a label.

**Anomaly detection** is the most common business application. After training an autoencoder on normal data — normal transactions, normal sensor readings, normal server logs — any input that is anomalous will reconstruct poorly because the encoder has never learned to compress it efficiently. A high reconstruction error flags the anomaly. This approach requires no labelled examples of anomalies, which is precisely the situation in most fraud and fault detection problems (anomalies are rare and often unseen during training).

Other practical uses include **dimensionality reduction** (a learned, non-linear alternative to PCA), **data denoising** (train on noisy inputs with clean targets), and **feature extraction** (use the encoder output as input to a downstream classifier).

## Architectures: FFN autoencoder and LSTM autoencoder

An autoencoder's encoder and decoder can be built from any architecture. The choice depends on the data type:

- For **tabular data** (customer transactions, sensor readings): FFN encoder + FFN decoder.
- For **images**: CNN encoder + transposed-convolution decoder.
- For **sequences** (time series, text): LSTM encoder + LSTM decoder.

In a **sequence autoencoder**, the LSTM encoder reads the full input sequence step by step and produces a single final hidden state — the bottleneck. The LSTM decoder then generates the reconstructed sequence from that hidden state, one step at a time. This is the architecture behind many time-series anomaly detection systems in production.

> **Why here, in Chapter 6?** Chapters 1–5 gave you the supervised learning toolkit: FFNs for tabular data, CNNs for images, LSTMs for sequences. Autoencoders are the natural bridge to *unsupervised* deep learning. They use the same building blocks you already know — `nn.Linear`, `nn.LSTM`, loss functions, the training loop — but turn the task around: instead of predicting a label, the network learns to explain the data itself. This concept also lays the groundwork for Chapter 7, where large language models use a form of self-supervised learning (predicting masked or next tokens) that is spiritually related to the autoencoder idea.


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# 6.6 — Autoencoder architecture illustration
#
# This cell defines both a tabular autoencoder (FFN-based) and a sequence
# autoencoder (LSTM-based). No training is run here — the goal is to see
# how the encoder-bottleneck-decoder structure is expressed in PyTorch.
#
# For a full anomaly detection worked example, see the Answer Key.
# ─────────────────────────────────────────────────────────────────────────────

# ── A. Tabular Autoencoder (FFN-based) ───────────────────────────────────────
#
# Use case: anomaly detection on transaction records.
# Train on normal transactions; flag high-reconstruction-error rows as anomalies.

class TabularAutoencoder(nn.Module):
    """Symmetric FFN autoencoder for tabular data.
    Encoder compresses input_dim → bottleneck_dim.
    Decoder reconstructs bottleneck_dim → input_dim.
    """
    def __init__(self, input_dim, bottleneck_dim=8):
        super().__init__()
        hidden = input_dim // 2
        self.encoder = nn.Sequential(
            nn.Linear(input_dim,    hidden),        nn.ReLU(),
            nn.Linear(hidden,       bottleneck_dim), # ← bottleneck
        )
        self.decoder = nn.Sequential(
            nn.Linear(bottleneck_dim, hidden),       nn.ReLU(),
            nn.Linear(hidden,         input_dim),    # ← reconstruction
        )

    def forward(self, x):
        z    = self.encoder(x)   # compressed representation
        x_hat = self.decoder(z)  # reconstruction
        return x_hat

    def encode(self, x):
        """Return latent vectors only — use as features for downstream tasks."""
        return self.encoder(x)


# ── B. Sequence Autoencoder (LSTM-based) ─────────────────────────────────────
#
# Use case: anomaly detection on time series (sensor readings, call volume).
# The encoder reads the sequence and produces a bottleneck hidden state.
# The decoder generates the reconstructed sequence from that hidden state.

class SequenceAutoencoder(nn.Module):
    """LSTM sequence autoencoder.
    Encoder  : LSTM reads input sequence → final hidden state h_n (bottleneck).
    Decoder  : LSTM generates reconstructed sequence from h_n.
    Loss     : MSE between original and reconstructed sequence.
    """
    def __init__(self, input_size=1, hidden_size=32, num_layers=1):
        super().__init__()
        self.hidden_size = hidden_size
        self.num_layers  = num_layers
        self.encoder_lstm = nn.LSTM(
            input_size, hidden_size, num_layers, batch_first=True
        )
        self.decoder_lstm = nn.LSTM(
            hidden_size, hidden_size, num_layers, batch_first=True
        )
        self.output_layer = nn.Linear(hidden_size, input_size)

    def forward(self, x):
        # x: (batch, seq_len, input_size)
        batch_size, seq_len, _ = x.shape

        # Encode — bottleneck is the final hidden state h_n
        _, (h_n, c_n) = self.encoder_lstm(x)

        # Decode — repeat the bottleneck vector across seq_len steps
        # then let the decoder LSTM generate the reconstruction
        decoder_input = h_n[-1].unsqueeze(1).repeat(1, seq_len, 1)  # (B, L, H)
        decoded, _    = self.decoder_lstm(decoder_input, (h_n, c_n))
        x_hat         = self.output_layer(decoded)  # (B, L, input_size)
        return x_hat


# ── Inspect parameter counts ──────────────────────────────────────────────────
tab_ae  = TabularAutoencoder(input_dim=20, bottleneck_dim=4)
seq_ae  = SequenceAutoencoder(input_size=1, hidden_size=32)

tab_params = sum(p.numel() for p in tab_ae.parameters())
seq_params = sum(p.numel() for p in seq_ae.parameters())

print('Tabular Autoencoder:')
print(tab_ae)
print(f'Parameters: {tab_params:,}  (input_dim=20, bottleneck=4)')
print()
print('Sequence Autoencoder:')
print(seq_ae)
print(f'Parameters: {seq_params:,}  (input_size=1, hidden=32)')
print()
print('Training recipe (same loop as every chapter):')
print('  criterion = nn.MSELoss()')
print('  loss      = criterion(model(X_batch), X_batch)  # target IS the input')
print('  At inference: flag rows where loss > threshold as anomalies.')


---
# 6.6 Pipeline: Saving and Deploying the Forecast Model

## Adapting ModelPipeline for sequence models

The `ModelPipeline` introduced in Chapter 3 and applied to FFNs (Chapter 4) and CNNs (Chapter 5) works the same way here — with one important addition for sequence models: the **window configuration** must be saved alongside the weights.

For an FFN, the preprocessor (a scaler) and the feature names are sufficient to reproduce predictions. For an LSTM forecast model, a caller also needs to know:

- `window_size` — how many past time steps to include in each input sequence
- `input_size` — how many features per time step (1 for univariate, more for multivariate)

These are saved in `model_config` alongside the architecture hyperparameters. `validate_input()` checks that incoming sequences have the correct shape `(batch, window_size, input_size)` before any inference is run.

**Table 6.6 — ModelPipeline for the LSTM forecast model**

| Field saved | Value | Purpose |
|------------|-------|----------|
| `model_class_bytes` | `dill.dumps(LSTMNet)` | Reconstruct the class on any machine |
| `model_config` | `{hidden_size, num_layers, window_size, ...}` | Rebuild the architecture + inform `validate_input()` |
| `state_dict` | Learned weights | The trained parameters |
| `preprocessor` | Fitted `MinMaxScaler` | Scales new data identically to training data |
| `training_history` | Loss curves | Diagnose and continue training |

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# ModelPipeline — Chapter 6 version
#
# Extended from Chapters 3–5 with sequence-aware validate_input():
#   - Checks tensor shape (batch, window_size, input_size)
#   - Reads window_size and input_size from model_config
# All other methods (save, load, retrain) are identical to previous chapters.
# ─────────────────────────────────────────────────────────────────────────────

import dill

class ModelPipeline:
    """Bundles a trained PyTorch model with its preprocessor and metadata.
    Provides: save | load | validate_input | predict | retrain.
    Chapter 6 extension: validate_input checks sequence shape for LSTM models.
    """

    def __init__(self, model, model_config, preprocessor=None,
                 feature_names=None, feature_ranges=None, model_class=None):
        self.model             = model
        self.model_config      = model_config
        self.preprocessor      = preprocessor
        self.feature_names     = feature_names or []
        self.feature_ranges    = feature_ranges or {}
        self._model_class      = model_class
        self.model_class       = model_class.__name__ if model_class else type(model).__name__
        self.training_history  = {'train_loss': [], 'val_loss': []}

    # ── Save ──────────────────────────────────────────────────────────────────
    def save(self, path):
        """Save the complete pipeline to a single .pth file."""
        cls_obj     = self._model_class if self._model_class is not None else type(self.model)
        class_bytes = dill.dumps(cls_obj)
        checkpoint  = {
            'model_class'       : self.model_class,
            'model_class_bytes' : class_bytes,
            'model_config'      : self.model_config,
            'state_dict'        : self.model.state_dict(),
            'preprocessor'      : self.preprocessor,
            'feature_names'     : self.feature_names,
            'feature_ranges'    : self.feature_ranges,
            'training_history'  : self.training_history,
            'pytorch_version'   : torch.__version__,
            'saved_at'          : datetime.datetime.now().isoformat(),
        }
        torch.save(checkpoint, path)
        print(f'Pipeline saved -> {path}')
        print(f'  model_class : {self.model_class}')
        print(f'  history     : {len(self.training_history["val_loss"])} epochs recorded')
        print(f'  saved_at    : {checkpoint["saved_at"]}')

    # ── Load ──────────────────────────────────────────────────────────────────
    @classmethod
    def load(cls, path, device=None):
        """Load a saved pipeline. No training code or class definition needed."""
        if device is None:
            device = torch.device('cpu')
        checkpoint  = torch.load(path, map_location=device)
        ModelClass  = dill.loads(checkpoint['model_class_bytes'])
        # Strip non-constructor keys before rebuilding the model
        arch_keys   = {'input_size', 'hidden_size', 'num_layers', 'dropout', 'output_size'}
        arch_config = {k: v for k, v in checkpoint['model_config'].items() if k in arch_keys}
        model       = ModelClass(**arch_config)
        model.load_state_dict(checkpoint['state_dict'])
        model.to(device).eval()
        pipeline = cls(
            model          = model,
            model_config   = checkpoint['model_config'],
            preprocessor   = checkpoint['preprocessor'],
            feature_names  = checkpoint['feature_names'],
            feature_ranges = checkpoint['feature_ranges'],
            model_class    = None,  # class already embedded in model
        )
        pipeline._model_class     = ModelClass
        pipeline.model_class      = checkpoint['model_class']
        pipeline.training_history = checkpoint['training_history']
        print(f'Pipeline loaded from {path}')
        print(f'  model_class : {pipeline.model_class}')
        print(f'  pytorch_ver : {checkpoint["pytorch_version"]}')
        print(f'  saved_at    : {checkpoint["saved_at"]}')
        return pipeline

    # ── Validate input ─────────────────────────────────────────────────────────
    def validate_input(self, X):
        """Check that X has the correct shape for this sequence model.
        X should be a tensor of shape (batch, window_size, input_size).
        Returns True if valid, raises ValueError if not.
        """
        expected_window = self.model_config.get('window_size')
        expected_feats  = self.model_config.get('input_size', 1)
        if not isinstance(X, torch.Tensor):
            raise ValueError('Input must be a torch.Tensor.')
        if X.ndim != 3:
            raise ValueError(f'Expected 3-D tensor (batch, window, features), got shape {X.shape}.')
        if expected_window is not None and X.shape[1] != expected_window:
            raise ValueError(
                f'Sequence length mismatch: model expects {expected_window} steps, '
                f'got {X.shape[1]}.'
            )
        if X.shape[2] != expected_feats:
            raise ValueError(
                f'Feature count mismatch: model expects {expected_feats} features per step, '
                f'got {X.shape[2]}.'
            )
        return True

    # ── Predict ────────────────────────────────────────────────────────────────
    def predict(self, X_raw, device=None):
        """Run inference on raw (unscaled) input sequences.
        X_raw: numpy array of shape (batch, window_size) or (batch, window_size, n_features).
        Returns a numpy array of predictions in original sales units.
        """
        if device is None:
            device = next(self.model.parameters()).device
        # Scale input
        if X_raw.ndim == 2:
            X_raw = X_raw[:, :, np.newaxis]      # (batch, window, 1)
        orig_shape = X_raw.shape
        flat = X_raw.reshape(-1, 1)
        scaled_flat = self.preprocessor.transform(flat)
        X_scaled    = scaled_flat.reshape(orig_shape).astype(np.float32)
        X_tensor    = torch.tensor(X_scaled)
        self.validate_input(X_tensor)
        self.model.eval()
        with torch.no_grad():
            preds_scaled = self.model(X_tensor.to(device)).cpu().numpy()
        # Inverse-transform predictions
        preds = self.preprocessor.inverse_transform(preds_scaled.reshape(-1, 1)).flatten()
        return preds

    # ── Retrain ────────────────────────────────────────────────────────────────
    def retrain(self, train_loader, val_loader, criterion, optimiser, num_epochs, device):
        """Continue training and append results to training_history."""
        self.model.train()
        new_train, new_val = run_training(
            self.model, train_loader, val_loader, criterion, optimiser,
            num_epochs=num_epochs, device=device, verbose_every=5,
        )
        self.training_history['train_loss'].extend(new_train)
        self.training_history['val_loss'].extend(new_val)
        print(f'Retrain complete. Total epochs in history: {len(self.training_history["val_loss"])}')


print('ModelPipeline defined.')
print('Methods: save | load | validate_input | predict | retrain')

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# 6.6a — Wrap the forecast model and save
# ─────────────────────────────────────────────────────────────────────────────

forecast_model_config = {
    **FORECAST_CONFIG,          # input_size, hidden_size, num_layers, dropout, output_size
    'window_size': WINDOW_SIZE, # saved so validate_input() can check incoming sequences
}

pipeline = ModelPipeline(
    model         = fc_model,
    model_config  = forecast_model_config,
    preprocessor  = scaler,        # fitted MinMaxScaler
    feature_names = ['total_sales'],
    model_class   = LSTMNet,
)
pipeline.training_history = {'train_loss': fc_train_hist, 'val_loss': fc_val_hist}

pipeline.save('forecast_model_v1.pth')
print(f'\nWindow size saved in model_config: {pipeline.model_config["window_size"]} days')
print('The scaler and window config are both inside the .pth file.')

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# 6.6b — Load in a clean context and forecast the next 28 days
#
# Multi-step forecasting: we feed the last WINDOW_SIZE days,
# predict 1 day ahead, append the prediction to the window, and repeat.
# This is called an autoregressive (recursive) forecasting strategy.
# ─────────────────────────────────────────────────────────────────────────────

del pipeline   # simulate a fresh session

loaded = ModelPipeline.load('forecast_model_v1.pth')
print(f'\nTraining history restored: {len(loaded.training_history["val_loss"])} epochs')

HORIZON = 28   # forecast 4 weeks ahead

# Seed: last WINDOW_SIZE days of actual sales (raw units)
seed_window = daily.total_sales.values[-WINDOW_SIZE:].copy()  # (WINDOW_SIZE,)
forecast    = []

for _ in range(HORIZON):
    pred_val = loaded.predict(seed_window[np.newaxis, :])  # (1,)
    forecast.append(pred_val[0])
    # Slide window forward: drop oldest value, append new prediction
    # predict() returns raw units; we need to append the scaled version
    new_scaled = loaded.preprocessor.transform([[pred_val[0]]])[0, 0]
    scaled_seed = loaded.preprocessor.transform(seed_window.reshape(-1, 1)).flatten()
    scaled_seed = np.append(scaled_seed[1:], new_scaled)
    seed_window = loaded.preprocessor.inverse_transform(scaled_seed.reshape(-1, 1)).flatten()

forecast = np.array(forecast)
last_date = daily.date.iloc[-1]
forecast_dates = pd.date_range(start=last_date + pd.Timedelta(days=1), periods=HORIZON)

# ── Figure 6.6 ────────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(12, 4))
fig.patch.set_facecolor('white')
ax.set_facecolor('white')

# Show the last 90 days of actual history as context
context_days  = daily.tail(90)
ax.plot(context_days.date, context_days.total_sales,
        color='#1F3864', lw=1.5, label='Actual (last 90 days)')
ax.plot(forecast_dates, forecast,
        color='#e74c3c', lw=2.0, linestyle='--', label=f'{HORIZON}-day forecast')
ax.axvline(x=last_date, color='#aaaaaa', lw=1.0, linestyle=':')
ax.set_ylabel('Total Units Sold', fontsize=10)
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:,.0f}'))
ax.set_title(f'Figure 6.6 — 28-Day Autoregressive Forecast (Loaded from .pth)',
             fontsize=11, fontweight='bold', color='#1F3864')
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()

print(f'\nForecast summary (next {HORIZON} days):')
print(f'  Mean : {forecast.mean():>12,.0f} units/day')
print(f'  Min  : {forecast.min():>12,.0f} units/day')
print(f'  Max  : {forecast.max():>12,.0f} units/day')

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# 6.6c — Retrain with one new month of sales data and save v2
#
# In production, a new month of sales would arrive from the data warehouse.
# Here we simulate it by using the most recent 30 days of actual data as
# "new" training examples.
# ─────────────────────────────────────────────────────────────────────────────

# Simulate new monthly data: take last 30 days of full series as new arrivals
new_data = daily.total_sales.values[-(WINDOW_SIZE + 30):]
X_new_raw = np.stack([
    new_data[i : i + WINDOW_SIZE]
    for i in range(len(new_data) - WINDOW_SIZE)
])                           # (30, WINDOW_SIZE)
y_new_raw = new_data[WINDOW_SIZE:]  # (30,)

# Scale using the loaded pipeline's scaler
X_new_sc = loaded.preprocessor.transform(X_new_raw.reshape(-1, 1)).reshape(30, WINDOW_SIZE)
y_new_sc = loaded.preprocessor.transform(y_new_raw.reshape(-1, 1)).reshape(-1, 1)

X_new_t = torch.tensor(X_new_sc, dtype=torch.float32).unsqueeze(-1)   # (30, W, 1)
y_new_t = torch.tensor(y_new_sc, dtype=torch.float32)                  # (30, 1)

new_loader = DataLoader(TensorDataset(X_new_t, y_new_t), batch_size=16, shuffle=True)

retrain_opt = optim.Adam(loaded.model.parameters(), lr=2e-4)

print('Retraining on new monthly data (10 additional epochs):')
loaded.retrain(
    train_loader = new_loader,
    val_loader   = fc_val_loader,
    criterion    = nn.MSELoss(),
    optimiser    = retrain_opt,
    num_epochs   = 10,
    device       = device,
)

loaded.save('forecast_model_v2.pth')
print(f'\nTotal epochs in history: {len(loaded.training_history["val_loss"])}')
print('Version 1 preserved for rollback if needed.')

### 📝 Exercise 6.6 — Pipeline for Sequence Models

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Exercise 6.6
#
# Task A: Load forecast_model_v1.pth and try to call validate_input() on a
#         tensor with the WRONG sequence length (e.g., 30 instead of 60).
#         Confirm that a ValueError is raised with an informative message.
#
# Task B: Load forecast_model_v2.pth and generate a 28-day forecast.
#         Plot both the v1 and v2 forecasts on the same axes.
#         Do the forecasts differ? Describe what changed.
#
# Task C: Answer the following as comments:
#   Q1. Why is the autoregressive (recursive) forecasting strategy above likely
#       to accumulate errors over long horizons?
#   Q2. What would you change to do direct multi-step forecasting
#       (predicting all 28 days in a single forward pass)?
#       (Hint: what would output_size and FORECAST_HORIZON be?)
# ─────────────────────────────────────────────────────────────────────────────

# Task A
# v1 = ModelPipeline.load('forecast_model_v1.pth')
# bad_input = torch.zeros(1, 30, 1)   # wrong seq length
# try:
#     v1.validate_input(bad_input)
# except ValueError as e:
#     print(f'Caught expected error: {e}')

# Task B
# v2 = ModelPipeline.load('forecast_model_v2.pth')
# ...

# See the Answer Key at the end of this chapter.
print('Exercise 6.6 complete.')

---
## Chapter 6 Summary

| Concept | Key takeaway |
|---------|-------------|
| **Why sequences are different** | Order matters — the meaning of each step depends on the steps that preceded it; FFNs and CNNs discard this dependency |
| **Hidden state** | A vector passed from step to step that acts as compressed memory of everything seen so far |
| **Vanilla RNN** | `nn.RNN` — simple and fast, but gradients vanish through the recurrent weight matrix; fails on sequences longer than ~20 steps |
| **Vanishing gradient (temporal)** | The same problem as in deep FFNs (Chapter 3) but in the time dimension — gradients are multiplied by $W_h$ at every step |
| **LSTM** | `nn.LSTM` — adds a cell state that flows additively across time; three gates (forget, input, output) learn what to keep, write, and expose |
| **Cell state vs. hidden state** | Cell state = long-term memory (additive updates); hidden state = short-term output (multiplicative, filtered through output gate) |
| **Gradient clipping** | `clip_grad_norm_(max_norm=1.0)` — best practice for all recurrent models; prevents exploding gradients in long sequences |
| **Sliding window** | Converts a flat time series into (X, y) pairs; `WINDOW_SIZE` is the look-back; temporal split is mandatory — a random split leaks the future into training |
| **MAE and MAPE** | MAE measures absolute error in original units; MAPE measures percentage error — MAPE is preferred when comparing across stores or products of different scale |
| **Autoregressive forecasting** | Predict one step ahead, append the prediction, repeat — errors accumulate; direct multi-step (`output_size=H`) avoids this at the cost of a harder learning problem |
| **Word embeddings** | `nn.Embedding` — learnable lookup table mapping integer token IDs to dense vectors; `padding_idx=0` prevents gradients from updating the padding entry |
| **Bag-of-words baseline** | Always establish before comparing LSTM results — if the LSTM cannot beat a logistic regression on word counts, diagnose before adding complexity |
| **Autoencoder** | Unsupervised architecture: encoder compresses input to a bottleneck; decoder reconstructs it; reconstruction error is a natural anomaly score |
| **Sequence autoencoder** | LSTM encoder → bottleneck hidden state → LSTM decoder; the input is both the model input and the training target |
| **ModelPipeline (sequence)** | Same save/load/retrain contract as Chapters 3–5; `model_config` stores `window_size` and `input_size`; `validate_input()` checks incoming tensor shape |

---

## What Comes Next

Chapter 7 introduces **Large Language Models** — the architecture that has transformed natural language processing across every industry. Where the sentiment LSTM in this chapter learned word meanings from scratch on 25,000 reviews, an LLM arrives pre-trained on hundreds of billions of tokens and can be directed through prompting alone. The chapter covers tokenisation, embeddings, the Transformer architecture that powers modern LLMs, practical use of LLM APIs (OpenAI, Gemini, Anthropic Claude), and a capstone project: a business Q&A bot that answers questions over a corpus of earnings call transcripts.

---
*Deep Learning for Business Analytics: From Basics to Large Language Models*  
*Dr. M. Ramasubramaniam &amp; Mr. Daniel Peter*

---
## Answer Key — Chapter 6 Exercises

Use these suggested answers to check your thinking. For open-ended exercises (6.1) there is no single correct answer — the sample responses show the kind of reasoning to aim for.

---

### Exercise 6.1 — Sequence Problems in Your Domain

*Open-ended — no single correct answer. Sample responses:*

1. **Hospital admissions (healthcare)**
   - Sequence: daily emergency admission counts over the past 90 days
   - Target: number of admissions next week, by ward
   - Why order matters: seasonal flu peaks, public-holiday patterns, and multi-day spikes after large events are all temporal patterns

2. **Customer support ticket threads (SaaS)**
   - Sequence: the ordered messages within a support conversation
   - Target: whether the ticket will escalate to a human agent
   - Why order matters: urgency accumulates — one frustrated message after several polite ones signals escalation risk

3. **Energy consumption (utilities)**
   - Sequence: hourly electricity demand over the past 7 days
   - Target: demand for the next 24 hours
   - Why order matters: morning and evening peaks repeat daily; the previous day's temperature influences the next day's baseline

---

### Exercise 6.2 — Observe RNN Failure on Long Sequences

**Q1 — Does a larger hidden size fix vanishing gradients?** No. A larger `W_h` adds more parameters, but the fundamental problem is that gradients are *multiplied* by `W_h` at every time step. Making the matrix larger does not change that multiplicative structure — it may even make exploding gradients more likely. You will find the large-hidden RNN on long sequences converges no better than the small one.

**Q2 — Is this a capacity problem or an architecture problem?** Architecture. A vanilla RNN lacks the structural mechanism to preserve gradient signal over long horizons. No amount of capacity (more neurons, more layers) can compensate for a design that shrinks gradients at every step. The fix is a different architecture — the LSTM.

```python
# Task: train a large RNN on the long-sequence dataset
torch.manual_seed(SEED)
rnn_large = VanillaRNN(hidden_size=128).to(device)
_, rnn_large_val = run_training(
    rnn_large, long_train_loader, long_val_loader, crit,
    optim.Adam(rnn_large.parameters(), lr=1e-3),
    num_epochs=NUM_EPOCHS_RNN, device=device, verbose_every=None,
)
print(f'RNN hidden=32   seq_len=60  final val: {rnn_long_val[-1]:.4f}')
print(f'RNN hidden=128  seq_len=60  final val: {rnn_large_val[-1]:.4f}')
# Both plateau near the same loss — capacity does not solve the vanishing gradient problem.
```

---

### Exercise 6.3 — Replace RNN with LSTM

**Q1 — Is LSTM better on short sequences?** On very short sequences (seq_len=10), the RNN and LSTM perform comparably — the vanishing gradient problem does not appear when there are only 10 multiplication steps. For short sequences the RNN is preferable: fewer parameters, faster to train, equally accurate. Use LSTM when seq_len > ~20.

**Q2 — How many more parameters does an LSTM have?** An LSTM has approximately **4×** the parameters of a vanilla RNN with the same hidden size, because each of the three gates plus the candidate cell state each requires its own full set of weight matrices (`W_x` and `W_h`).

```python
# Task: train an LSTMNet on the short-sequence dataset and compare
torch.manual_seed(SEED)
lstm_short = LSTMNet(input_size=1, hidden_size=32, num_layers=1, dropout=0.0).to(device)
_, lstm_short_val = run_training(
    lstm_short, short_train_loader, short_val_loader, crit,
    optim.Adam(lstm_short.parameters(), lr=1e-3),
    num_epochs=NUM_EPOCHS_RNN, device=device, verbose_every=None,
)
print(f'RNN  hidden=32 seq_len=10: {rnn_short_val[-1]:.4f}')
print(f'LSTM hidden=32 seq_len=10: {lstm_short_val[-1]:.4f}')

rnn_params  = sum(p.numel() for p in VanillaRNN(hidden_size=32).parameters())
lstm_params = sum(p.numel() for p in LSTMNet(input_size=1, hidden_size=32,
                                              num_layers=1, dropout=0.0).parameters())
print(f'RNN params: {rnn_params:,}   LSTM params: {lstm_params:,}')
print(f'Ratio: {lstm_params / rnn_params:.1f}x')
```

---

### Exercise 6.4 — End-to-End Demand Forecast

**Task A — shorter window (WINDOW_SIZE=30):**
```python
WINDOW_SIZE_A = 30
X_a, y_a     = make_windows(scaled, WINDOW_SIZE_A, FORECAST_HORIZON)
split_a      = split_idx - WINDOW_SIZE_A
tr_a = DataLoader(TensorDataset(X_a[:split_a], y_a[:split_a]), batch_size=64, shuffle=True)
va_a = DataLoader(TensorDataset(X_a[split_a:], y_a[split_a:]), batch_size=64)

torch.manual_seed(SEED)
cfg_a   = {**FORECAST_CONFIG, 'input_size': 1}
model_a = LSTMNet(**cfg_a).to(device)
run_training(model_a, tr_a, va_a, nn.MSELoss(),
             optim.Adam(model_a.parameters(), lr=1e-3),
             num_epochs=50, device=device, verbose_every=10)
# Shorter windows typically hurt MAE on trend-heavy series — the model loses
# the medium-term context that drives most of the predictable signal.
```

**Task B — add day-of-week as a second feature:**
```python
daily['dow']     = daily.date.dt.dayofweek       # 0 = Monday
dow_scaled       = daily.dow.values / 6.0        # scale to [0, 1]
two_feature_arr  = np.stack([scaled.flatten(), dow_scaled], axis=1)  # (N, 2)

# make_windows must be updated to handle 2-D input:
def make_windows_2d(arr, window_size, horizon=1):
    X, y = [], []
    for i in range(len(arr) - window_size - horizon + 1):
        X.append(arr[i : i + window_size])            # (W, 2)
        y.append(arr[i + window_size, 0:1])           # predict sales only
    return (torch.tensor(np.array(X), dtype=torch.float32),
            torch.tensor(np.array(y), dtype=torch.float32))

# Then set input_size=2 in FORECAST_CONFIG before training.
```

**Q1 — Why prefer MAPE over MAE?** MAE is in original sales units — a value of 400,000 units/day is hard to judge without knowing the typical volume. MAPE expresses error as a percentage, making it comparable across stores with very different throughput. Most retail forecasting SLAs are specified as "≤X% MAPE" for this reason. MAPE breaks down when actual values are near zero (division instability).

**Q2 — Why is a random split wrong for time series?** A random split assigns some future dates to the training set. The model effectively sees the future during training — data leakage. In deployment, future data is never available. A temporal split faithfully replicates the deployment scenario: train on everything before date *d*, evaluate on dates after *d*.

---

### Exercise 6.5 — Sentiment Classifier

**Task A — smaller embedding (embed_dim=32):**
```python
SENTIMENT_CONFIG_A = {**SENTIMENT_CONFIG, 'embed_dim': 32}
torch.manual_seed(SEED)
sent_model_a = SentimentLSTM(**SENTIMENT_CONFIG_A).to(device)
# Train with same loop as Section 6.5e.
# Accuracy typically drops 1-3% — the smaller embedding space cannot capture
# fine-grained semantic distinctions, but for binary sentiment it is often sufficient.
```

**Task B — longer context (MAX_LEN=512):**
```python
MAX_LEN = 512
# Re-run 6.5b to re-encode reviews with the longer limit, then retrain.
# IMDB reviews are typically 200-400 tokens; extending to 512 rarely changes
# accuracy meaningfully but doubles memory usage and training time.
```

**Q1 — When does LSTM beat bag-of-words?** Two cases where order matters most: (a) **negation** — "I would not recommend this" has the same bag-of-words representation as "I would recommend this, not bad at all" but opposite sentiment; (b) **contrastive reviews** — "The acting is superb but the plot is a complete mess" — the final clause often determines the verdict, and an LSTM can learn to weight it more heavily.

**Q2 — Advantage of pre-trained embeddings?** Pre-trained embeddings (GloVe, word2vec) encode semantic similarity learned from billions of tokens. With only 25,000 reviews, many vocabulary words appear too rarely for the model to learn a reliable embedding from scratch. Pre-trained vectors give rare words a reasonable starting point, reducing the labelled data needed to achieve good performance.

---

### Exercise 6.6 — Pipeline for Sequence Models

**Task A — validate wrong sequence length:**
```python
v1 = ModelPipeline.load('forecast_model_v1.pth')
bad_input = torch.zeros(1, 30, 1)   # wrong: model expects window_size=60
try:
    v1.validate_input(bad_input)
except ValueError as e:
    print(f'Caught expected error: {e}')
# Output: Sequence length mismatch: model expects 60 steps, got 30.
```

**Task B — compare v1 vs v2 forecasts:**
```python
v2 = ModelPipeline.load('forecast_model_v2.pth')
# Run the same 28-day autoregressive loop as Section 6.6b for both v1 and v2.
# Plot both forecast series on the same axes.
# The v2 forecast will differ slightly — the 10 additional epochs on new monthly
# data shift the weights toward more recent sales patterns.
```

**Q1 — Why do autoregressive errors accumulate?** Each predicted value is fed back as input for the next step. An error at step *t+1* propagates into step *t+2*, compounding across the 28-day horizon. On noisy or trend-changing series, the forecast can drift significantly by week 4.

**Q2 — How to switch to direct multi-step forecasting?** Set `output_size=28` in `FORECAST_CONFIG` and `FORECAST_HORIZON=28` in `make_windows`. The model predicts all 28 future values in a single forward pass. `MSELoss` then averages over all 28 output positions. This eliminates error compounding but is a harder learning problem — the single hidden state must encode enough information to produce 28 distinct predictions simultaneously.


---
*Deep Learning for Business Analytics: From Basics to Large Language Models*  
*Dr. M. Ramasubramaniam &amp; Mr. Daniel Peter*